Домашку будет легче делать в колабе (убедитесь, что у вас runtype с gpu).

# Задание 1 (3 балла)

Обучите word2vec модели с негативным семплированием (cbow и skip-gram) аналогично тому, как это было сделано в семинаре. Вам нужно изменить следующие пункты:
1) добавьте лемматизацию в предобработку (любым способом)  
2) измените размер окна в большую или меньшую сторону
3) измените размерность итоговых векторов

Выберете несколько не похожих по смыслу слов (не таких как в семинаре), и протестируйте полученные эмбединги (найдите ближайшие слова и оцените качество, как в семинаре).
Постарайтесь обучать модели как можно дольше и на как можно большем количестве данных. (Но если у вас мало времени или ресурсов, то допустимо взять поменьше данных и поставить меньше эпох)

In [1]:
! pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 53.1 MB/s eta 0:00:00


In [2]:
import re
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter
from string import punctuation
from sklearn.model_selection import train_test_split
import pymorphy3

morph = pymorphy3.MorphAnalyzer()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [3]:
wiki = open('wiki_data.txt').read().split('\n')

In [4]:
def preprocess_with_lemma(text):
    """Предобработка текста с лемматизацией"""
    tokens = re.sub('#+', ' ', text.lower()).split()
    tokens = [token.strip(punctuation) for token in tokens]
    tokens = [token for token in tokens if token and len(token) > 1]

    lemmas = []
    for token in tokens:
        parsed = morph.parse(token)[0]
        lemmas.append(parsed.normal_form)

    return lemmas

vocab = Counter()

for i, text in enumerate(wiki):
    vocab.update(preprocess_with_lemma(text))

print(f"Всего уникальных слов: {len(vocab)}")



Всего уникальных слов: 262504


In [5]:
min_count = 30
filtered_vocab = {word for word in vocab if vocab[word] > min_count}
print(f"Слов после фильтрации (>{min_count}): {len(filtered_vocab)}")

word2id = {'PAD': 0}
for word in filtered_vocab:
    word2id[word] = len(word2id)

id2word = {i: word for word, i in word2id.items()}
vocab_size = len(word2id)
print(f"Размер словаря: {vocab_size}")

Слов после фильтрации (>30): 12397
Размер словаря: 12398


In [6]:
sentences = []

for text in wiki:
    tokens = preprocess_with_lemma(text)
    if not tokens:
        continue
    ids = [word2id[token] for token in tokens if token in word2id]
    if len(ids) > 2:
        sentences.append(ids)

print(f"Всего предложений: {len(sentences)}")

Всего предложений: 19416


In [7]:
def pad_sequences(sequences, maxlen, padding='post', value=0):
    res = np.full((len(sequences), maxlen), value, dtype='int64')
    for i, seq in enumerate(sequences):
        if not seq:
            continue
        if len(seq) >= maxlen:
            if padding == 'post':
                res[i] = np.array(seq[:maxlen])
            else:
                res[i] = np.array(seq[-maxlen:])
        else:
            if padding == 'post':
                res[i, :len(seq)] = np.array(seq)
            else:
                res[i, -len(seq):] = np.array(seq)
    return res

In [8]:
def most_similar(word, embeddings, top_n=10):
    if word not in word2id:
        print(f"Слово '{word}' не найдено в словаре")
        return []

    word_idx = word2id[word]
    word_vec = embeddings[word_idx]

    word_vec_norm = word_vec / (np.linalg.norm(word_vec) + 1e-8)
    emb_norms = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8)

    similarities = np.dot(emb_norms, word_vec_norm)

    top_indices = np.argsort(similarities)[::-1][1:top_n+1]

    results = [(id2word[idx], similarities[idx]) for idx in top_indices]

    print(f"\nНаиболее похожие слова для '{word}':")
    for w, sim in results:
        print(f"  {w}: {sim:.4f}")

    return results

In [9]:
WINDOW_SIZE = 3
EMB_DIM = 150

def gen_batches_skipgram(sentences, window=3, batch_size=1024, num_negatives=5):
    """Генератор батчей для Skip-gram с негативным семплированием"""
    while True:
        X_target = []
        X_context = []
        y = []

        for sent in sentences:
            for i in range(len(sent)):
                word = sent[i]
                start = max(0, i - window)
                end = min(len(sent), i + window + 1)
                context = sent[start:i] + sent[i+1:end]

                for context_word in context:
                    X_target.append(word)
                    X_context.append(context_word)
                    y.append(1)

                    for _ in range(num_negatives):
                        neg_word = np.random.randint(1, vocab_size)  # Исключаем PAD
                        X_target.append(word)
                        X_context.append(neg_word)
                        y.append(0)

                    if len(X_target) >= batch_size:
                        yield (np.array(X_target, dtype='int64'),
                               np.array(X_context, dtype='int64'),
                               np.array(y, dtype='float32'))
                        X_target, X_context, y = [], [], []


class SkipGramNegSampling(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.target_emb = nn.Embedding(vocab_size, emb_dim)
        self.context_emb = nn.Embedding(vocab_size, emb_dim)

        nn.init.xavier_uniform_(self.target_emb.weight)
        nn.init.xavier_uniform_(self.context_emb.weight)

    def forward(self, target_ids, context_ids):
        """
        target_ids: (batch,)
        context_ids: (batch,)
        """
        t = self.target_emb(target_ids)
        c = self.context_emb(context_ids)
        dot = (t * c).sum(dim=1)
        return dot

In [11]:
model_sg = SkipGramNegSampling(vocab_size, EMB_DIM).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_sg.parameters(), lr=1e-3)

train_size = int(len(sentences) * 0.9)
train_sentences = sentences[:train_size]
valid_sentences = sentences[train_size:]

train_gen = gen_batches_skipgram(train_sentences, window=WINDOW_SIZE, batch_size=2048)
valid_gen = gen_batches_skipgram(valid_sentences, window=WINDOW_SIZE, batch_size=2048)

num_epochs = 20
steps_per_epoch = 3000
validation_steps = 100

train_losses_sg = []
valid_losses_sg = []

for epoch in range(num_epochs):
    model_sg.train()
    epoch_loss = 0.0

    for step in range(steps_per_epoch):
        X_t, X_c, y = next(train_gen)
        X_t = torch.LongTensor(X_t).to(device)
        X_c = torch.LongTensor(X_c).to(device)
        y_t = torch.FloatTensor(y).to(device)

        optimizer.zero_grad()
        logits = model_sg(X_t, X_c)
        loss = criterion(logits, y_t)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= steps_per_epoch
    train_losses_sg.append(epoch_loss)

    model_sg.eval()
    val_loss = 0.0
    with torch.no_grad():
        for _ in range(validation_steps):
            X_t, X_c, y = next(valid_gen)
            X_t = torch.LongTensor(X_t).to(device)
            X_c = torch.LongTensor(X_c).to(device)
            y_t = torch.FloatTensor(y).to(device)

            logits = model_sg(X_t, X_c)
            loss = criterion(logits, y_t)
            val_loss += loss.item()

    val_loss /= validation_steps
    valid_losses_sg.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} - train loss: {epoch_loss:.4f}, val loss: {val_loss:.4f}")

Epoch 1/20 - train loss: 0.3818, val loss: 0.3240
Epoch 2/20 - train loss: 0.3070, val loss: 0.2815
Epoch 3/20 - train loss: 0.2980, val loss: 0.2960
Epoch 4/20 - train loss: 0.3032, val loss: 0.3253
Epoch 5/20 - train loss: 0.2982, val loss: 0.3101
Epoch 6/20 - train loss: 0.2887, val loss: 0.2894
Epoch 7/20 - train loss: 0.2955, val loss: 0.3071
Epoch 8/20 - train loss: 0.2964, val loss: 0.2788
Epoch 9/20 - train loss: 0.2541, val loss: 0.2876
Epoch 10/20 - train loss: 0.2899, val loss: 0.2993
Epoch 11/20 - train loss: 0.2905, val loss: 0.3008
Epoch 12/20 - train loss: 0.2862, val loss: 0.2795
Epoch 13/20 - train loss: 0.2860, val loss: 0.3091
Epoch 14/20 - train loss: 0.2878, val loss: 0.2739
Epoch 15/20 - train loss: 0.2813, val loss: 0.3012
Epoch 16/20 - train loss: 0.2807, val loss: 0.2790
Epoch 17/20 - train loss: 0.2851, val loss: 0.3027
Epoch 18/20 - train loss: 0.2850, val loss: 0.2889
Epoch 19/20 - train loss: 0.2747, val loss: 0.3004
Epoch 20/20 - train loss: 0.2789, val lo

In [12]:
emb_target_sg = model_sg.target_emb.weight.detach().cpu().numpy()
emb_context_sg = model_sg.context_emb.weight.detach().cpu().numpy()
embeddings_sg = (emb_target_sg + emb_context_sg) / 2

In [13]:
test_words = ['компьютер', 'математика', 'война', 'музыка', 'океан', 'президент']

print("ТЕСТИРОВАНИЕ SKIP-GRAM")
for word in test_words:
    most_similar(word, embeddings_sg, top_n=8)

ТЕСТИРОВАНИЕ SKIP-GRAM

Наиболее похожие слова для 'компьютер':
  бесплатный: 0.9465
  microsoft: 0.9463
  процессор: 0.9445
  цифровой: 0.9437
  playstation: 0.9431
  консоль: 0.9418
  движок: 0.9411
  windows: 0.9394

Наиболее похожие слова для 'математика':
  химия: 0.9662
  физика: 0.9604
  астрономия: 0.9579
  физико-математический: 0.9548
  философский: 0.9541
  доцент: 0.9537
  педагогический: 0.9514
  выпускник: 0.9505

Наиболее похожие слова для 'война':
  в: 0.6051
  мировой: 0.5921
  второй: 0.5824
  время: 0.5657
  великий: 0.5499
  период: 0.5495
  начало: 0.5346
  армия: 0.5343

Наиболее похожие слова для 'музыка':
  музыкальный: 0.8451
  опера: 0.7732
  классический: 0.7716
  произведение: 0.7681
  композитор: 0.7651
  творчество: 0.7427
  жанр: 0.7254
  сцена: 0.7247

Наиболее похожие слова для 'океан':
  тихий: 0.9442
  атлантический: 0.9178
  пролив: 0.8783
  индийский: 0.8756
  ледовитый: 0.8746
  мыс: 0.8734
  залив: 0.8679
  бухта: 0.8642

Наиболее похожие слова дл

In [14]:
def gen_batches_cbow(sentences, window=3, batch_size=1024, num_negatives=5):
    """Генератор батчей для CBOW с негативным семплированием"""
    while True:
        X_target = []
        X_context = []
        y = []

        for sent in sentences:
            for i in range(len(sent)):
                word = sent[i]
                start = max(0, i - window)
                end = min(len(sent), i + window + 1)
                context = sent[start:i] + sent[i+1:end]

                if not context:
                    continue

                X_target.append(word)
                X_context.append(context)
                y.append(1)

                for _ in range(num_negatives):
                    neg_word = np.random.randint(1, vocab_size)
                    X_target.append(neg_word)
                    X_context.append(context)
                    y.append(0)

                if len(X_target) >= batch_size:
                    X_context_pad = pad_sequences(X_context, maxlen=window*2, value=0)
                    yield (np.array(X_target, dtype='int64'),
                           X_context_pad,
                           np.array(y, dtype='float32'))
                    X_target, X_context, y = [], [], []


class CBOWNegSampling(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.target_emb = nn.Embedding(vocab_size, emb_dim)
        self.context_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        nn.init.xavier_uniform_(self.target_emb.weight)
        nn.init.xavier_uniform_(self.context_emb.weight)

    def forward(self, target_ids, context_ids):
        """
        target_ids: (batch,)
        context_ids: (batch, context_size)
        """
        t = self.target_emb(target_ids)  # (batch, emb_dim)
        c = self.context_emb(context_ids)  # (batch, context_size, emb_dim)

        mask = (context_ids != 0).float().unsqueeze(-1)  # (batch, context_size, 1)
        c = c * mask

        c_mean = c.sum(dim=1) / (mask.sum(dim=1) + 1e-8)  # (batch, emb_dim)

        dot = (t * c_mean).sum(dim=1)  # (batch,)
        return dot

In [15]:
model_cbow = CBOWNegSampling(vocab_size, EMB_DIM).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_cbow.parameters(), lr=1e-3)

train_gen_cbow = gen_batches_cbow(train_sentences, window=WINDOW_SIZE, batch_size=2048)
valid_gen_cbow = gen_batches_cbow(valid_sentences, window=WINDOW_SIZE, batch_size=2048)

train_losses_cbow = []
valid_losses_cbow = []

for epoch in range(num_epochs):
    model_cbow.train()
    epoch_loss = 0.0

    for step in range(steps_per_epoch):
        X_t, X_c, y = next(train_gen_cbow)
        X_t = torch.LongTensor(X_t).to(device)
        X_c = torch.LongTensor(X_c).to(device)
        y_t = torch.FloatTensor(y).to(device)

        optimizer.zero_grad()
        logits = model_cbow(X_t, X_c)
        loss = criterion(logits, y_t)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= steps_per_epoch
    train_losses_cbow.append(epoch_loss)

    model_cbow.eval()
    val_loss = 0.0
    with torch.no_grad():
        for _ in range(validation_steps):
            X_t, X_c, y = next(valid_gen_cbow)
            X_t = torch.LongTensor(X_t).to(device)
            X_c = torch.LongTensor(X_c).to(device)
            y_t = torch.FloatTensor(y).to(device)

            logits = model_cbow(X_t, X_c)
            loss = criterion(logits, y_t)
            val_loss += loss.item()

    val_loss /= validation_steps
    valid_losses_cbow.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} - train loss: {epoch_loss:.4f}, val loss: {val_loss:.4f}")


Epoch 1/20 - train loss: 0.3282, val loss: 0.2917
Epoch 2/20 - train loss: 0.2818, val loss: 0.2697
Epoch 3/20 - train loss: 0.2564, val loss: 0.2458
Epoch 4/20 - train loss: 0.2334, val loss: 0.2320
Epoch 5/20 - train loss: 0.2164, val loss: 0.2172
Epoch 6/20 - train loss: 0.2105, val loss: 0.2211
Epoch 7/20 - train loss: 0.2001, val loss: 0.1991
Epoch 8/20 - train loss: 0.1909, val loss: 0.2056
Epoch 9/20 - train loss: 0.1913, val loss: 0.2023
Epoch 10/20 - train loss: 0.1826, val loss: 0.2005
Epoch 11/20 - train loss: 0.1779, val loss: 0.2106
Epoch 12/20 - train loss: 0.1768, val loss: 0.1909
Epoch 13/20 - train loss: 0.1717, val loss: 0.1857
Epoch 14/20 - train loss: 0.1681, val loss: 0.1835
Epoch 15/20 - train loss: 0.1639, val loss: 0.1863
Epoch 16/20 - train loss: 0.1621, val loss: 0.1904
Epoch 17/20 - train loss: 0.1579, val loss: 0.1817
Epoch 18/20 - train loss: 0.1562, val loss: 0.1913
Epoch 19/20 - train loss: 0.1560, val loss: 0.1747
Epoch 20/20 - train loss: 0.1510, val lo

In [16]:
emb_target_cbow = model_cbow.target_emb.weight.detach().cpu().numpy()
emb_context_cbow = model_cbow.context_emb.weight.detach().cpu().numpy()
embeddings_cbow = (emb_target_cbow + emb_context_cbow) / 2

In [17]:
print("ТЕСТИРОВАНИЕ CBOW")
for word in test_words:
    most_similar(word, embeddings_cbow, top_n=8)

ТЕСТИРОВАНИЕ CBOW

Наиболее похожие слова для 'компьютер':
  интерфейс: 0.7974
  процессор: 0.7872
  ibm: 0.7685
  виртуальный: 0.7611
  плеер: 0.7575
  встроить: 0.7397
  intel: 0.7352
  персональный: 0.7325

Наиболее похожие слова для 'математика':
  химия: 0.8461
  физика: 0.8350
  астрономия: 0.8302
  психология: 0.8155
  педагогика: 0.7997
  языкознание: 0.7990
  филология: 0.7954
  лингвистика: 0.7917

Наиболее похожие слова для 'война':
  русско-турецкий: 0.6014
  гражданский: 0.5832
  советско-финский: 0.5705
  великий: 0.5493
  1812: 0.5128
  мировой: 0.5099
  1877—1878: 0.5098
  отечественный: 0.5027

Наиболее похожие слова для 'музыка':
  музыкальный: 0.7498
  джазовый: 0.7171
  вокальный: 0.7035
  композитор: 0.7026
  танец: 0.6909
  хор: 0.6883
  хоровой: 0.6863
  репертуар: 0.6842

Наиболее похожие слова для 'океан':
  тихий: 0.8703
  атлантический: 0.8309
  ледовитый: 0.7902
  индийский: 0.7239
  акватория: 0.6374
  пролив: 0.6190
  залив: 0.6121
  широта: 0.6065

Наибол

# Задание 2 (2 балла)

Обучите 1 word2vec и 1 fastext модель в gensim. В каждой из модели нужно задать все параметры, которые мы разбирали на семинаре. Заданные значения должны отличаться от дефолтных и от тех, что мы использовали на семинаре.

In [18]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 69.9 MB/s eta 0:00:00


In [19]:
import gensim
from gensim.models import Word2Vec, FastText

In [20]:
texts = []
for sent_ids in sentences:
    sent_words = [id2word[idx] for idx in sent_ids if idx in id2word]
    if len(sent_words) > 2:
        texts.append(sent_words)

print(f"Подготовлено {len(texts)} текстов для Gensim")
print(f"Пример: {texts[0][:10]}")

Подготовлено 19416 текстов для Gensim
Пример: ['нижегородский', 'область', 'сельский', 'посёлок', 'дивеевский', 'район', 'нижегородский', 'область', 'входить', 'состав']


In [23]:
w2v_model = Word2Vec(
    sentences=texts,
    vector_size=150,
    min_count=20,
    max_vocab_size=9000,
    window=7,
    epochs=8,
    sg=1,
    hs=0,
    negative=10,
    sample=1e-4,
    ns_exponent=0.65,
    alpha=0.03,
    min_alpha=0.0005,
    seed=42
)


In [25]:

print("ТЕСТИРОВАНИЕ WORD2VEC")

for word in test_words:
    if word in w2v_model.wv:
        print(f"\nПохожие слова для '{word}':")
        similar = w2v_model.wv.most_similar(word, topn=7)
        for w, score in similar:
            print(f"  {w}: {score:.4f}")
    else:
        print(f"\nСлово '{word}' не найдено в словаре")

ТЕСТИРОВАНИЕ WORD2VEC

Похожие слова для 'компьютер':
  процессор: 0.6735
  microsoft: 0.6673
  windows: 0.6376
  pc: 0.6369
  клавиатура: 0.6117
  компьютерный: 0.6044
  intel: 0.5963

Похожие слова для 'математика':
  математический: 0.7146
  физика: 0.6808
  логика: 0.6752
  физико-математический: 0.6497
  университет: 0.6381
  философия: 0.6324
  прикладной: 0.6264

Похожие слова для 'война':
  отечественный: 0.6573
  воевать: 0.6299
  военный: 0.6294
  мировой: 0.6218
  великий: 0.5988
  армия: 0.5961
  оккупировать: 0.5808

Похожие слова для 'музыка':
  музыкальный: 0.8009
  композитор: 0.7809
  вокальный: 0.7443
  джазовый: 0.7222
  репертуар: 0.7154
  исполнитель: 0.6969
  песня: 0.6895

Похожие слова для 'океан':
  атлантический: 0.7404
  тихий: 0.7360
  антарктида: 0.6566
  атлантика: 0.6478
  море: 0.5950
  меридиан: 0.5850
  акватория: 0.5670

Похожие слова для 'президент':
  президентский: 0.6903
  премьер-министр: 0.6442
  вице-президент: 0.6388
  правительство: 0.6179
  

In [26]:
ft_model = FastText(
    sentences=texts,
    vector_size=250,
    min_count=10,
    max_vocab_size=10000,
    window=6,
    epochs=8,
    sg=0,
    hs=0,
    negative=8,
    sample=5e-5,
    ns_exponent=0.7,
    cbow_mean=1,
    min_n=4,
    max_n=7,
    bucket=1500000,
    alpha=0.025,
    min_alpha=0.0002,
    seed=42
)

In [27]:
print("ТЕСТИРОВАНИЕ FASTTEXT")
for word in test_words:
    if word in ft_model.wv:
        print(f"\nПохожие слова для '{word}':")
        similar = ft_model.wv.most_similar(word, topn=7)
        for w, score in similar:
            print(f"  {w}: {score:.4f}")
    else:
        print(f"\nСлово '{word}' не найдено в словаре")

oov_words = ['программированию', 'университетский', 'кинематографический',
             'микропроцессор', 'суперкомпьютер']

for word in oov_words:
    try:
        similar = ft_model.wv.most_similar(word, topn=5)
        print(f"\nПохожие слова для редких слов '{word}':")
        for w, score in similar:
            print(f"  {w}: {score:.4f}")
    except KeyError:
        print(f"\nНе удалось обработать '{word}'")

ТЕСТИРОВАНИЕ FASTTEXT

Похожие слова для 'компьютер':
  компьютерный: 0.9158
  движок: 0.9085
  windows: 0.9014
  ibm: 0.9000
  microsoft: 0.8953
  software: 0.8829
  интерфейс: 0.8775

Похожие слова для 'математика':
  математик: 0.9753
  физика: 0.9252
  физик: 0.9038
  теоретик: 0.8978
  тематика: 0.8862
  химия: 0.8584
  теория: 0.8380

Похожие слова для 'война':
  мировой: 0.7656
  в: 0.7576
  оккупация: 0.7347
  вели: 0.7067
  великий: 0.6915
  вермахт: 0.6915
  послевоенный: 0.6842

Похожие слова для 'музыка':
  музыкант: 0.9224
  композитор: 0.8740
  репертуар: 0.8641
  музыкальный: 0.8530
  фортепиано: 0.8337
  труппа: 0.8285
  певец: 0.8266

Похожие слова для 'океан':
  тихий: 0.9041
  антарктида: 0.8830
  побережье: 0.8783
  атлантика: 0.8558
  море: 0.8524
  ледовитый: 0.8343
  пролив: 0.8294

Похожие слова для 'президент':
  вице-президент: 0.9486
  президентский: 0.9101
  кнессет: 0.8310
  ассамблея: 0.8120
  кандидатура: 0.7748
  премьер-министр: 0.7676
  парламент: 0.76

# Задание 3 (3 балла)

Используя датасет для классификации (labeled.csv), обучите классификатор на базе эмбеддингов. Оцените качество на отложенной выборке.   
В качестве эмбеддинг модели вы можете использовать одну из моделей обученных в предыдущем задании или использовать одну из предобученных моделей с rusvectores (удостоверьтесь что правильно воспроизводите предобработку в этом случае!)  
Для того, чтобы построить эмбединг целого текста, усредните вектора отдельных слов в один общий вектор.
В качестве алгоритма классификации используйте LogisicticRegression (можете попробовать SGDClassifier, чтобы было побыстрее)  
F1 мера должна быть выше 20%.

In [28]:
import pandas as pd

In [29]:
df = pd.read_csv('labeled (1).csv')

In [30]:
df

,comment,toxic
0,"Верблюдов-то за что? Дебилы, бл...\n",1.0
1,"Хохлы, это отдушина затюканого россиянина, мол...",1.0
2,Собаке - собачья смерть\n,1.0
3,"Страницу обнови, дебил. Это тоже не оскорблени...",1.0
4,"тебя не убедил 6-страничный пдф в том, что Скр...",1.0
...,...,...
14407,Вонючий совковый скот прибежал и ноет. А вот и...,1.0
14408,А кого любить? Гоблина тупорылого что-ли? Или ...,1.0
14409,"Посмотрел Утомленных солнцем 2. И оказалось, ч...",0.0
14410,КРЫМОТРЕД НАРУШАЕТ ПРАВИЛА РАЗДЕЛА Т.К В НЕМ Н...,1.0


In [31]:
from tqdm import tqdm

In [32]:
def text_to_embedding(text, model):
    tokens = preprocess_with_lemma(text)
    emb_dim = model.wv.vector_size

    if not tokens:
        return np.zeros(emb_dim)

    vectors = []
    for token in tokens:
        try:
            vectors.append(model.wv[token])
        except KeyError:
            continue

    if not vectors:
        return np.zeros(emb_dim)

    return np.mean(vectors, axis=0)

print("Создание эмбеддингов...")
emb_dim = ft_model.wv.vector_size

X = np.zeros((len(df), emb_dim))
for i, text in enumerate(tqdm(df['comment'])):
    X[i] = text_to_embedding(text, ft_model)

y = df['toxic'].values

print(f"\nРазмер X: {X.shape}")
print(f"Размер y: {y.shape}")


Создание эмбеддингов...


100%|██████████| 14412/14412 [00:47<00:00, 303.26it/s]


Размер X: (14412, 250)
Размер y: (14412,)


In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [36]:

from sklearn.linear_model import LogisticRegression, SGDClassifier

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    C=1.0,
    solver='lbfgs'
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


In [38]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix
print("РЕЗУЛЬТАТЫ ")
print(f"\nF1 Score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1 Score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1 Score (binary/toxic): {f1_score(y_test, y_pred, average='binary'):.4f}")

print(classification_report(y_test, y_pred, target_names=['Non-toxic', 'Toxic']))


РЕЗУЛЬТАТЫ 

F1 Score (macro): 0.7670
F1 Score (weighted): 0.7873
F1 Score (binary/toxic): 0.7056
              precision    recall  f1-score   support

   Non-toxic       0.87      0.79      0.83      1918
       Toxic       0.65      0.78      0.71       965

    accuracy                           0.78      2883
   macro avg       0.76      0.78      0.77      2883
weighted avg       0.80      0.78      0.79      2883



# Задание 4 (2 доп балла)

В тетрадку с фастекстом добавьте код для обучения с negative sampling (задача сводится к бинарной классификации) и обучите модель. Проверьте полученную модель на нескольких словах. Похожие слова должны быть похожими по смыслу и по форме.

In [40]:
def tokenize(text):
    tokens = re.sub('#+', ' ', text.lower()).split()
    tokens = [token.strip(punctuation) for token in tokens]
    tokens = [token for token in tokens if token]
    return tokens

def ngrammer(raw_string, n=2):
    ngrams = []
    raw_string = ''.join(['<', raw_string, '>'])
    for i in range(0, len(raw_string)-n+1):
        ngram = ''.join(raw_string[i:i+n])
        if ngram == '<' or ngram == '>':
            continue
        ngrams.append(ngram)
    return ngrams

def split_tokens(tokens, min_ngram_size, max_ngram_size):
    tokens_with_subwords = []
    for token in tokens:
        subtokens = []
        for i in range(min_ngram_size, max_ngram_size+1):
            if len(token) > i:
                subtokens.extend(ngrammer(token, i))
        tokens_with_subwords.append(subtokens)
    return tokens_with_subwords


class SubwordTokenizer:
    def __init__(self, ngram_range=(1,1), min_count=5):
        self.min_ngram_size, self.max_ngram_size = ngram_range
        self.min_count = min_count
        self.subword_vocab = None
        self.fullword_vocab = None
        self.vocab = None
        self.id2word = None
        self.word2id = None

    def build_vocab(self, texts):
        unfiltered_subword_vocab = Counter()
        unfiltered_fullword_vocab = Counter()
        for text in texts:
            tokens = tokenize(text)
            unfiltered_fullword_vocab.update(tokens)
            subwords_per_token = split_tokens(tokens, self.min_ngram_size, self.max_ngram_size)
            for subwords in subwords_per_token:
                unfiltered_subword_vocab.update(set(subwords))

        self.fullword_vocab = set()
        self.subword_vocab = set()

        for word, count in unfiltered_fullword_vocab.items():
            if count >= self.min_count:
                self.fullword_vocab.add(word)
        for word, count in unfiltered_subword_vocab.items():
            if count >= (self.min_count * 100):
                self.subword_vocab.add(word)

        self.vocab = self.fullword_vocab | self.subword_vocab
        self.id2word = {i: word for i, word in enumerate(self.vocab)}
        self.word2id = {word: i for i, word in self.id2word.items()}

        self.fullword_list = list(self.fullword_vocab)
        self.fullword_ids = [self.word2id[w] for w in self.fullword_list]

    def subword_tokenize(self, text):
        if self.vocab is None:
            raise AttributeError('Vocabulary is not built!')
        tokens = tokenize(text)
        tokens_with_subwords = split_tokens(tokens, self.min_ngram_size, self.max_ngram_size)
        only_vocab_tokens_with_subwords = []
        for full_token, sub_tokens in zip(tokens, tokens_with_subwords):
            filtered = []
            if full_token in self.vocab:
                filtered.append(full_token)
            filtered.extend([subtoken for subtoken in set(sub_tokens) if subtoken in self.vocab])
            only_vocab_tokens_with_subwords.append(filtered)
        return only_vocab_tokens_with_subwords

    def encode(self, subword_tokenized_text):
        encoded_text = []
        for token in subword_tokenized_text:
            if not token:
                continue
            encoded_text.append([self.word2id[token[0]]] +
                              [self.word2id[t] for t in set(token[1:])
                               if t in self.word2id and t != token[0]])
        return encoded_text

    def __call__(self, text):
        return self.encode(self.subword_tokenize(text))

tokenizer = SubwordTokenizer(ngram_range=(2, 4), min_count=10)
tokenizer.build_vocab(wiki)


In [41]:
print(f"Размер полного словаря: {len(tokenizer.vocab)}")
print(f"Полных слов: {len(tokenizer.fullword_vocab)}")
print(f"Подслов (n-грамм): {len(tokenizer.subword_vocab)}")

Размер полного словаря: 54860
Полных слов: 46798
Подслов (n-грамм): 9242


In [42]:
def pad_sequences(sequences, maxlen, padding='post', value=0):
    """Паддинг последовательностей"""
    res = np.full((len(sequences), maxlen), value, dtype='int64')
    for i, seq in enumerate(sequences):
        if not seq:
            continue
        if len(seq) >= maxlen:
            if padding == 'post':
                res[i] = np.array(seq[:maxlen])
            else:
                res[i] = np.array(seq[-maxlen:])
        else:
            if padding == 'post':
                res[i, :len(seq)] = np.array(seq)
            else:
                res[i, -len(seq):] = np.array(seq)
    return res


def gen_batches_ft_negative_sampling(sentences, tokenizer, window=5,
                                      batch_size=1000, maxlen=20,
                                      num_negatives=5):
    left_context_length = int(np.ceil(window / 2))
    right_context_length = window // 2

    fullword_ids = np.array(tokenizer.fullword_ids)
    num_fullwords = len(fullword_ids)

    while True:
        X_word = []
        X_context = []
        y = []

        for sent in sentences:
            sent_encoded = tokenizer(sent)

            if len(sent_encoded) < 2:
                continue

            for i in range(len(sent_encoded)):
                word_with_subtokens = sent_encoded[i]

                if not word_with_subtokens:
                    continue

                start = max(0, i - left_context_length)
                end = min(len(sent_encoded), i + right_context_length + 1)
                context_indices = list(range(start, i)) + list(range(i + 1, end))

                for j in context_indices:
                    if not sent_encoded[j]:
                        continue

                    context_word_id = sent_encoded[j][0]

                    X_word.append(word_with_subtokens)
                    X_context.append(context_word_id)
                    y.append(1)

                    neg_indices = np.random.randint(0, num_fullwords, size=num_negatives)
                    neg_word_ids = fullword_ids[neg_indices]

                    for neg_id in neg_word_ids:
                        X_word.append(word_with_subtokens)
                        X_context.append(neg_id)
                        y.append(0)

                    if len(X_word) >= batch_size:
                        X_word_arr = pad_sequences(X_word, maxlen=maxlen, padding='post', value=0)
                        X_context_arr = np.array(X_context, dtype='int64')
                        y_arr = np.array(y, dtype='float32')

                        yield X_word_arr, X_context_arr, y_arr

                        X_word = []
                        X_context = []
                        y = []

In [43]:
class FastTextNegSampling(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.word_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.context_emb = nn.Embedding(vocab_size, emb_dim)
        nn.init.xavier_uniform_(self.word_emb.weight)
        nn.init.xavier_uniform_(self.context_emb.weight)
        self.word_emb.weight.data[0].fill_(0)

    def forward(self, word_ids, context_ids):
        word_emb = self.word_emb(word_ids)  # (batch, maxlen, emb_dim)
        mask = (word_ids != 0).float().unsqueeze(-1)  # (batch, maxlen, 1)
        word_emb_masked = word_emb * mask
        word_vec = word_emb_masked.sum(dim=1) / (mask.sum(dim=1) + 1e-8)  # (batch, emb_dim)
        context_vec = self.context_emb(context_ids)  # (batch, emb_dim)
        dot = (word_vec * context_vec).sum(dim=1)  # (batch,)

        return dot

    def get_word_embedding(self, word_ids):
        word_emb = self.word_emb(word_ids)
        mask = (word_ids != 0).float().unsqueeze(-1)
        word_emb_masked = word_emb * mask
        word_vec = word_emb_masked.sum(dim=1) / (mask.sum(dim=1) + 1e-8)
        return word_vec

In [46]:
vocab_size = len(tokenizer.vocab)
emb_dim = 100
window = 5
num_negatives = 5
batch_size = 512
maxlen = 20

model = FastTextNegSampling(vocab_size, emb_dim).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_size = int(len(wiki) * 0.9)
train_gen = gen_batches_ft_negative_sampling(
    wiki[:train_size], tokenizer,
    window=window, batch_size=batch_size,
    maxlen=maxlen, num_negatives=num_negatives
)
valid_gen = gen_batches_ft_negative_sampling(
    wiki[train_size:], tokenizer,
    window=window, batch_size=batch_size,
    maxlen=maxlen, num_negatives=num_negatives
)

steps_per_epoch = 5000
validation_steps = 100
num_epochs = 10

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # TRAIN
    model.train()
    epoch_loss = 0.0

    for step in range(steps_per_epoch):
        X_word, X_context, y_batch = next(train_gen)

        X_word = torch.LongTensor(X_word).to(device)
        X_context = torch.LongTensor(X_context).to(device)
        y_batch = torch.FloatTensor(y_batch).to(device)

        optimizer.zero_grad()
        logits = model(X_word, X_context)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= steps_per_epoch
    train_losses.append(epoch_loss)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for _ in range(validation_steps):
            X_word, X_context, y_batch = next(valid_gen)

            X_word = torch.LongTensor(X_word).to(device)
            X_context = torch.LongTensor(X_context).to(device)
            y_batch = torch.FloatTensor(y_batch).to(device)

            logits = model(X_word, X_context)
            loss = criterion(logits, y_batch)
            val_loss += loss.item()

    val_loss /= validation_steps
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} - train loss: {epoch_loss:.4f}, val loss: {val_loss:.4f}")


Epoch 1/10 - train loss: 0.3043, val loss: 0.2542
Epoch 2/10 - train loss: 0.2555, val loss: 0.2459
Epoch 3/10 - train loss: 0.2578, val loss: 0.2140
Epoch 4/10 - train loss: 0.2534, val loss: 0.2190
Epoch 5/10 - train loss: 0.2557, val loss: 0.2146
Epoch 6/10 - train loss: 0.2595, val loss: 0.2635
Epoch 7/10 - train loss: 0.2274, val loss: 0.2415
Epoch 8/10 - train loss: 0.2504, val loss: 0.2350
Epoch 9/10 - train loss: 0.2542, val loss: 0.2565
Epoch 10/10 - train loss: 0.2541, val loss: 0.2597


In [48]:
from sklearn.metrics.pairwise import cosine_distances

embeddings = model.word_emb.weight.detach().cpu().numpy()
id2word_fullwords = list(tokenizer.fullword_vocab)
full_word_embeddings = np.zeros((len(id2word_fullwords), emb_dim))

for i, word in enumerate(id2word_fullwords):
    subword_ids = tokenizer(word)[0]
    if subword_ids:
        full_word_embeddings[i] = embeddings[subword_ids].mean(axis=0)

def most_similar_ft(word, embeddings, tokenizer, top_n=15):
    subwords = tokenizer(word)

    if not subwords or not subwords[0]:
        print(f"Слово '{word}' не найдено в словаре")
        return []

    subword_ids = subwords[0]
    word_embedding = embeddings[subword_ids].mean(axis=0)
    distances = cosine_distances(word_embedding.reshape(1, -1), full_word_embeddings)[0]
    sorted_indices = distances.argsort()
    results = []
    for idx in sorted_indices:
        similar_word = id2word_fullwords[idx]
        if similar_word != word:
            results.append((similar_word, 1 - distances[idx]))
        if len(results) >= top_n:
            break

    return results


def print_similar(word, embeddings, tokenizer, top_n=10):
    print(f"Похожие слова для: '{word}'")

    results = most_similar_ft(word, embeddings, tokenizer, top_n)

    if not results:
        return

    for i, (w, sim) in enumerate(results, 1):
        print(f"{i:2d}. {w:20s} (similarity: {sim:.4f})")

In [49]:
for word in test_words:
    print_similar(word, embeddings, tokenizer, top_n=10)

Похожие слова для: 'компьютер'
 1. компанией            (similarity: 0.9741)
 2. компакт-диск         (similarity: 0.9710)
 3. компьютера           (similarity: 0.9700)
 4. компания»            (similarity: 0.9691)
 5. компаний             (similarity: 0.9627)
 6. компаньони           (similarity: 0.9609)
 7. комплект             (similarity: 0.9595)
 8. компания             (similarity: 0.9595)
 9. авиакомпанией        (similarity: 0.9589)
10. композицией          (similarity: 0.9571)
Похожие слова для: 'математика'
 1. симмонса             (similarity: 0.9729)
 2. грамматика           (similarity: 0.9726)
 3. свидетелем           (similarity: 0.9633)
 4. тема                 (similarity: 0.9615)
 5. филаменты            (similarity: 0.9611)
 6. тематика             (similarity: 0.9600)
 7. ударника             (similarity: 0.9588)
 8. ватикана             (similarity: 0.9571)
 9. соратника            (similarity: 0.9566)
10. вадим                (similarity: 0.9565)
Похожие слова для

In [54]:
def get_oov_embedding(word, embeddings, tokenizer):
    ngrams = []
    for n in range(tokenizer.min_ngram_size, tokenizer.max_ngram_size + 1):
        ngrams.extend(ngrammer(word, n))
    known_ngrams = [ng for ng in ngrams if ng in tokenizer.word2id]

    if not known_ngrams:
        return None
    ngram_ids = [tokenizer.word2id[ng] for ng in known_ngrams]
    return embeddings[ngram_ids].mean(axis=0)


def most_similar_oov(word, embeddings, tokenizer, top_n=10):
    word_emb = get_oov_embedding(word, embeddings, tokenizer)

    if word_emb is None:
        print(f"Не удалось создать эмбеддинг для '{word}'")
        return []

    distances = cosine_distances(word_emb.reshape(1, -1), full_word_embeddings)[0]
    sorted_indices = distances.argsort()[:top_n]

    results = [(id2word_fullwords[idx], 1 - distances[idx]) for idx in sorted_indices]
    return results

oov_words = [
    'микропроцессор',
    'кинематографист',
]

for word in oov_words:
    if word in tokenizer.fullword_vocab:
        print(f"\n'{word}' есть в словаре, используем обычный поиск:")
        print_similar(word, embeddings, tokenizer, top_n=5)
    else:
        print(f"\n'{word}' - OOV слово, используем n-граммы:")
        results = most_similar_oov(word, embeddings, tokenizer, top_n=5)
        for i, (w, sim) in enumerate(results, 1):
            print(f"{i:2d}. {w:20s} (similarity: {sim:.4f})")


'микропроцессор' - OOV слово, используем n-граммы:
 1. процесса             (similarity: 0.9641)
 2. психиатр             (similarity: 0.9581)
 3. процессор            (similarity: 0.9553)
 4. психиатрия           (similarity: 0.9552)
 5. микробиологии        (similarity: 0.9549)

'кинематографист' - OOV слово, используем n-граммы:
 1. кинематограф         (similarity: 0.9831)
 2. кинематографистов    (similarity: 0.9783)
 3. монографий           (similarity: 0.9776)
 4. кинематографе        (similarity: 0.9728)
 5. фотографий           (similarity: 0.9721)
